# 02 — Encoding Robustness Check
## Is the Logistic Regression collapse an artefact of label encoding?

Reproduces **Table 4** of the manuscript.

---

**The concern.** The paper's argument rests on the linear model: LR attains perfect separation on the uncontrolled feature set (a shortcut), then collapses to 0.9566 when four protocol-identity fields are removed — while Random Forest stays at zero errors. If that collapse were merely an artefact of *label-encoding* six categorical fields as integers (which imposes a spurious ordinal structure that a linear model is uniquely sensitive to), the inference would not survive.

**The test.** Refit LR under **one-hot** encoding on both feature sets, with the encoder, scaler, and classifier all fitted on the **training partition only** — which additionally forecloses any preprocessing-leakage objection.

**The result.** The two encodings agree to four decimal places:

| Encoding | Uncontrolled (48) | Leakage-controlled (44) | Δ |
|---|---|---|---|
| Label | 1.0000 | 0.9566 | −0.0434 |
| One-hot | 1.0000 | 0.9566 | −0.0434 |

The perfect linear separability is a property of the **feature set**, not of the encoding applied to it. The inference stands.


In [ ]:
from google.colab import files
uploaded = files.upload()

CSV_PATH = list(uploaded.keys())[0]
print(f"\nUpload complete: {CSV_PATH}")

## Step 1-ALT — Use Google Drive instead (only if the upload above failed)

Put `ML-EdgeIIoT-dataset.csv` in the top level of your Google Drive, then run this cell and authorise access when prompted.

In [ ]:
# ONLY run this cell if the upload in Step 1 did not work.
# from google.colab import drive
# drive.mount('/content/drive')
# CSV_PATH = '/content/drive/MyDrive/ML-EdgeIIoT-dataset.csv'
# print('Using:', CSV_PATH)

## Step 2 — Load and clean

**Checkpoint:** this cell prints the row count after removing nulls and exact duplicates. It must read **152,245** — the figure in your manuscript. If it does not, stop and report the number before going further.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

RANDOM_STATE = 42

df = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols")

# --- CRITICAL: drop the 13 identifier/payload fields FIRST, then deduplicate ---
DROP_13 = [
    "frame.time", "ip.src_host", "ip.dst_host",
    "arp.dst.proto_ipv4", "arp.src.proto_ipv4",
    "http.file_data", "http.request.uri.query", "http.request.full_uri",
    "http.referer", "tcp.options", "tcp.payload", "tcp.srcport", "mqtt.msg",
]
df = df.drop(columns=[c for c in DROP_13 if c in df.columns])
print(f"After dropping 13 fields: {df.shape[1]} cols remain")

df = df.dropna().drop_duplicates().reset_index(drop=True)
print(f"\nAfter dropna + drop_duplicates: {df.shape[0]:,} rows")

assert len(df) == 152245, f"Expected 152,245 rows, got {len(df):,}"
print("CHECKPOINT PASSED — matches the manuscript.")

y_bin   = df["Attack_label"].astype(int)
y_strat = df["Attack_type"]
X_all   = df.drop(columns=["Attack_label", "Attack_type"])

LEAKAGE_FIELDS = ["dns.qry.name.len", "mqtt.topic", "mqtt.protoname", "mqtt.conack.flags"]
CATEGORICAL    = ["http.request.method", "http.request.version", "dns.qry.name.len",
                  "mqtt.conack.flags", "mqtt.protoname", "mqtt.topic"]

FEATURE_SETS = {
    "Uncontrolled (48)":       list(X_all.columns),
    "Leakage-controlled (44)": [c for c in X_all.columns if c not in LEAKAGE_FIELDS],
}
for k, v in FEATURE_SETS.items():
    print(f"  {k}: {len(v)} features")


## Step 3 — Define the feature sets

Drops the same 13 fields as the main study, then defines the uncontrolled (48-feature) and leakage-controlled (44-feature) sets.

## Step 4 — Run the four models

Logistic Regression × {label, one-hot} × {48 features, 44 features}.

Encoders and scaler are fitted on the **training split only** — this also closes any scaler-leakage objection.

This takes a few minutes. The one-hot runs are slower.

In [ ]:
def run_lr(cols, encoding):
    X = X_all[cols].copy()
    cat_cols = [c for c in CATEGORICAL if c in cols]
    num_cols = [c for c in cols if c not in cat_cols]

    Xtr, Xte, ytr, yte = train_test_split(
        X, y_bin, test_size=0.20, random_state=RANDOM_STATE, stratify=y_strat
    )

    if encoding == "onehot":
        pre = ColumnTransformer([
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_cols),
            ("num", StandardScaler(), num_cols),
        ])
    else:
        for c in cat_cols:
            cats = pd.concat([Xtr[c], Xte[c]]).astype("category").cat.categories
            Xtr[c] = pd.Categorical(Xtr[c], categories=cats).codes
            Xte[c] = pd.Categorical(Xte[c], categories=cats).codes
        pre = ColumnTransformer([("all", StandardScaler(), cols)])

    clf = Pipeline([
        ("pre", pre),
        ("lr", LogisticRegression(penalty="l2", max_iter=2000, random_state=RANDOM_STATE)),
    ])
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    prob = clf.predict_proba(Xte)[:, 1]

    return {
        "Accuracy":  accuracy_score(yte, pred),
        "Precision": precision_score(yte, pred),
        "Recall":    recall_score(yte, pred),
        "F1":        f1_score(yte, pred),
        "AUC":       roc_auc_score(yte, prob),
        "n_test":    len(yte),
    }

rows = []
for enc in ["label", "onehot"]:
    for name, cols in FEATURE_SETS.items():
        print(f"Running: LR | {enc:<6} | {name} ...")
        m = run_lr(cols, enc)
        rows.append({"Encoding": enc, "Feature set": name, **m})
        print(f"   accuracy={m['Accuracy']:.4f}   AUC={m['AUC']:.4f}\n")

res = pd.DataFrame(rows)
print("Done.")

## Step 5 — Results

The **one-hot delta** is the number this whole check exists to produce. Copy the output of this cell and send it back.

In [ ]:
print("=" * 78)
print("LOGISTIC REGRESSION -- ENCODING ROBUSTNESS CHECK")
print("=" * 78)
print(res.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print("\nDELTA (uncontrolled -> leakage-controlled), by encoding:")
for enc in ["label", "onehot"]:
    sub = res[res.Encoding == enc].set_index("Feature set")
    a = sub.loc["Uncontrolled (48)", "Accuracy"]
    b = sub.loc["Leakage-controlled (44)", "Accuracy"]
    print(f"  {enc:<6}:  {a:.4f} -> {b:.4f}   (delta {b - a:+.4f})")

res.to_csv("lr_encoding_robustness.csv", index=False)
print("\nSaved -> lr_encoding_robustness.csv")

## Step 6 — Download the results file (optional)

In [ ]:
from google.colab import files
files.download("lr_encoding_robustness.csv")